# Estudo de Caso 10.5 — Condensador a Ar (ACC) com Tubos Aletados

| Campo | Detalhe |
|:------|:--------|
| **Capítulo Associado** | Capítulo 10 — Aletas e Superfícies Estendidas |
| **Nível** | Pós-Graduação |
| **Tempo estimado** | 120 minutos |
| **Autor** | Jader Lugon Junior |
| **Método numérico** | Solução Analítica (Funções de Bessel) + Método ε-NTU |
| **Bibliografia** | Incropera et al. (2017), Cap. 3 e 11; Briggs & Young (1963); Gobbi (s.d.), Cap. 3 |

---

## 📋 Resumo

Neste estudo de caso, dimensionamos e verificamos o desempenho térmico de uma célula de um **Condensador a Ar (ACC — Air Cooled Condenser)** para uma usina termelétrica de 100 MW. O ar atmosférico possui coeficiente convectivo muito baixo ($h \sim 30$–$150$ W/m²K) comparado à condensação de vapor ($h \sim 5000$–$15000$ W/m²K), exigindo o uso de **tubos aletados** para compensar o desequilíbrio na rede de resistências térmicas. Implementamos a solução analítica exata (Funções de Bessel) para eficiência de aletas anulares, construímos a rede de resistências e verificamos o desempenho pelo Método ε-NTU.

---

## 🎯 Objetivos de Aprendizagem

Ao final deste estudo de caso, o leitor será capaz de:

1. **Aplicar** a solução analítica exata (Funções de Bessel) para eficiência de aletas anulares
2. **Utilizar** correlações empíricas de literatura (Briggs & Young) para coeficiente convectivo em feixes aletados
3. **Construir** a rede de resistências térmicas e calcular o coeficiente global $U_o$
4. **Verificar** o desempenho do trocador utilizando o Método ε-NTU para mudança de fase
5. **Comparar** a solução de Bessel com a aproximação de aleta reta, quantificando o erro
6. **Analisar** o impacto de extremos climáticos (ondas de calor) na potência da usina
7. **Avaliar** trade-offs entre condensação a ar e a água sob a ótica ambiental

---

In [ ]:
# ============================================================
# SEÇÃO 0: CONFIGURAÇÃO INICIAL
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
from scipy.special import iv, kv  # Funções de Bessel modificadas
from IPython.display import display, Markdown
import warnings
warnings.filterwarnings('ignore')

# Configuração de estilo dos gráficos
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.size'] = 12
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Ambiente configurado com sucesso!")
print(f"📦 NumPy {np.__version__} | Matplotlib {plt.matplotlib.__version__}")

---

## 🌊 Seção 1: Parâmetros do Projeto e Propriedades Termofísicas

A usina possui 24 células idênticas. Cada célula deve rejeitar **6,25 MW** de carga térmica. Os parâmetros são organizados em três grupos: fluido quente (vapor condensante), fluido frio (ar atmosférico) e geometria dos tubos/aletas.

| Parâmetro | Símbolo | Valor | Unidade |
|:----------|:--------|:------|:--------|
| Temperatura de saturação | $T_{sat}$ | 55,0 | °C |
| Calor latente | $h_{fg}$ | 2370 × 10³ | J/kg |
| Temperatura de entrada do ar | $T_{ar,in}$ | 30,0 | °C |
| Temperatura de saída do ar | $T_{ar,out}$ | 45,0 | °C |
| Velocidade frontal do ar | $u_{ar}$ | 3,0 | m/s |
| Diâmetro interno do tubo | $D_i$ | 0,021 | m |
| Diâmetro externo do tubo | $D_o$ | 0,025 | m |
| Raio da base da aleta | $r_1$ | 0,0125 | m |
| Raio do topo da aleta | $r_2$ | 0,025 | m |
| Espessura da aleta | $t$ | 0,0004 | m |
| Densidade de aletas | $N_f$ | 400 | aletas/m |
| Comprimento do tubo | $L$ | 9,0 | m |
| Número de tubos por célula | $N_t$ | 630 | — |

> 📖 **Referência:** Incropera et al. (2017), Cap. 3, Seção 3.6.4; Briggs & Young (1963).

In [ ]:
# ============================================================
# PARÂMETROS DO PROJETO — UMA CÉLULA DO ACC INDUSTRIAL
# ============================================================

# --- Fluido Quente (Vapor condensando DENTRO dos tubos) ---
mdot_v   = 63.3 / 24         # Vazão de vapor por célula [kg/s]
T_sat    = 55.0              # Temperatura de saturação [°C]
h_fg     = 2370e3            # Calor latente [J/kg]
rho_l    = 985.0             # Densidade do líquido [kg/m³]
rho_v    = 0.104             # Densidade do vapor [kg/m³]
mu_l     = 5.08e-4           # Viscosidade dinâmica do líquido [Pa·s]
k_l      = 0.652             # Condutividade do líquido [W/(m·K)]

# --- Fluido Frio (Ar atmosférico sobre as aletas) ---
T_ar_in  = 30.0              # Temperatura de entrada do ar [°C]
T_ar_out = 45.0              # Temperatura de saída do ar [°C]
rho_ar   = 1.10              # Densidade do ar [kg/m³]
cp_ar    = 1005.0            # Calor específico do ar [J/(kg·K)]
mu_ar    = 1.95e-5           # Viscosidade dinâmica do ar [Pa·s]
k_ar     = 0.028             # Condutividade térmica do ar [W/(m·K)]
Pr_ar    = 0.71              # Número de Prandtl do ar [-]
u_ar     = 3.0               # Velocidade frontal do ar [m/s]

# --- Geometria (Tubos de Aço / Aletas de Alumínio) ---
D_i      = 0.021             # Diâmetro interno do tubo [m]
D_o      = 0.025             # Diâmetro externo do tubo [m]
k_tubo   = 50.0              # Condutividade do aço carbono [W/(m·K)]
L_tubo   = 9.0               # Comprimento do tubo [m]
N_t      = 630               # Número de tubos por célula [-]
r_1      = D_o / 2           # Raio da base da aleta [m]
r_2      = 0.025             # Raio do topo da aleta [m]
t_aleta  = 0.0004            # Espessura da aleta [m]
N_aletas = 400               # Densidade de aletas [aletas/m]
k_aleta  = 205.0             # Condutividade do alumínio [W/(m·K)]

# --- Incrustações (Fouling) ---
Rf_i     = 0.0001            # Resistência de incrustação interna [m²·K/W]
Rf_o     = 0.0003            # Resistência de incrustação externa [m²·K/W]

# --- Coeficiente convectivo interno (condensação) ---
# Valor típico para condensação de vapor em tubos horizontais
h_i      = 8000.0            # Coeficiente de condensação [W/(m²·K)]

print("=" * 60)
print(" PARÂMETROS DO PROJETO — ACC")
print("=" * 60)
print(f"\n📊 Fluido Quente (Vapor):")
print(f"  • T_sat = {T_sat} °C")
print(f"  • h_fg = {h_fg/1e6:.2f} MJ/kg")
print(f"  • h_i = {h_i} W/(m²·K)")
print(f"\n🌬️ Fluido Frio (Ar):")
print(f"  • T_ar: {T_ar_in} → {T_ar_out} °C")
print(f"  • u_ar = {u_ar} m/s")
print(f"\n📐 Geometria:")
print(f"  • D_i = {D_i*1000:.1f} mm | D_o = {D_o*1000:.1f} mm")
print(f"  • r_1 = {r_1*1000:.2f} mm | r_2 = {r_2*1000:.1f} mm")
print(f"  • N_t = {N_t} tubos | L = {L_tubo} m")
print(f"  • N_aletas = {N_aletas} aletas/m")

---

## 🔬 Seção 2: Verificação — Balanço Energético

Antes de prosseguir com o dimensionamento, verificamos a consistência dos dados através da Primeira Lei da Termodinâmica. O calor liberado pela condensação deve ser integralmente absorvido pelo ar:

$$Q = \dot{m}_v \, h_{fg} = \dot{m}_{ar} \, c_{p,ar} \, (T_{ar,out} - T_{ar,in})$$

> 📖 **Referência:** Incropera et al. (2017), Cap. 11, Eq. 11.6.

In [ ]:
# ============================================================
# VERIFICAÇÃO: BALANÇO ENERGÉTICO
# ============================================================

# Carga térmica da condensação
Q_total = mdot_v * h_fg  # [W]

# Vazão mássica de ar necessária
mdot_ar = Q_total / (cp_ar * (T_ar_out - T_ar_in))  # [kg/s]

# Vazão volumétrica de ar
V_dot_ar = mdot_ar / rho_ar  # [m³/s]

# Área frontal necessária
A_frontal = V_dot_ar / u_ar  # [m²]

print("=" * 60)
print(" VERIFICAÇÃO: BALANÇO ENERGÉTICO")
print("=" * 60)
print(f"\n🧮 Cálculo:")
print(f"  • Q = ṁ_v × h_fg = {mdot_v:.3f} × {h_fg/1e6:.2f}×10⁶")
print(f"  • Q = {Q_total/1e6:.2f} MW")
print(f"\n📊 Resultados:")
print(f"  • Carga térmica: Q = {Q_total/1e6:.2f} MW")
print(f"  • Vazão mássica de ar: ṁ_ar = {mdot_ar:.1f} kg/s")
print(f"  • Vazão volumétrica: V̇ = {V_dot_ar:.1f} m³/s")
print(f"  • Área frontal: A_f = {A_frontal:.1f} m²")

# Verificação: a carga deve ser ≈ 6.25 MW
erro_balanco = abs(Q_total/1e6 - 6.25) / 6.25 * 100
assert erro_balanco < 5.0, f"Erro de {erro_balanco:.1f}% no balanço!"
print(f"\n✅ Balanço verificado (erro = {erro_balanco:.2f}% < 5%)")

---

## 📐 Seção 3: Formulação Matemática

### 3.1 Equação de Bessel para Aletas Anulares

A distribuição de temperatura em uma aleta anular é governada pela equação de Bessel modificada. A eficiência teórica é:

$$\eta_f = \frac{2\,r_1}{m\,(r_2^2 - r_1^2)} \cdot \frac{I_1(mr_2)\,K_1(mr_1) - K_1(mr_2)\,I_1(mr_1)}{I_0(mr_1)\,K_1(mr_2) + K_0(mr_1)\,I_1(mr_2)}$$

onde:
- $I_n$, $K_n$ = funções de Bessel modificadas de 1ª e 2ª espécie
- $m = \sqrt{\frac{2\,h_o}{k_{aleta}\,t}}$ = parâmetro da aleta [m⁻¹]
- $r_1$, $r_2$ = raios da base e do topo da aleta [m]

### 3.2 Rede de Resistências Térmicas

$$R_{total} = R_{int} + R_{f,i} + R_{parede} + R_{f,o} + R_{ext}$$

onde:
- $R_{int} = \frac{1}{h_i \, A_i}$ — convecção interna (condensação)
- $R_{f,i} = \frac{R_{f,i}}{A_i}$ — incrustação interna
- $R_{parede} = \frac{\ln(D_o/D_i)}{2\pi\,k_{tubo}\,L\,N_t}$ — condução na parede
- $R_{f,o} = \frac{R_{f,o}}{A_o}$ — incrustação externa
- $R_{ext} = \frac{1}{\eta_o\,h_o\,A_o}$ — convecção externa (aletada)

### 3.3 Eficiência de Superfície Total

$$\eta_o = 1 - \frac{A_{aletas}}{A_o}(1 - \eta_f)$$

### 3.4 Método ε-NTU (para condensação, $C_r = 0$)

$$\varepsilon = 1 - e^{-NTU}, \quad NTU = \frac{U_o \, A_o}{C_{ar}}$$

> 📖 **Referência:** Incropera et al. (2017), Cap. 3, Eq. 3.86; Cap. 11, Tabela 11.3.

---

## ⚡ Seção 4: Critério de Validade e Consistência

Neste estudo de caso, não se aplica uma condição CFL (não há equação diferencial transitória). Em vez disso, verificamos:

1. **Validade da solução de Bessel**: O parâmetro $m(r_2 - r_1)$ deve estar em faixa fisicamente razoável ($0 < m \cdot L_c < 5$)
2. **Regime de escoamento**: $Re_D$ deve estar na faixa de validade da correlação de Briggs & Young
3. **Dominância térmica**: A resistência do lado do ar deve dominar (> 70% do total)

> 💡 **Observação:** Para problemas de transferência de calor em regime permanente, a "estabilidade" é substituída pela verificação de consistência física dos parâmetros.

In [ ]:
# ============================================================
# CRITÉRIO DE VALIDADE E CONSISTÊNCIA
# ============================================================

# Número de Reynolds do ar baseado no diâmetro externo
Re_D = rho_ar * u_ar * D_o / mu_ar

# Áreas por metro de tubo
A_tubo_base = np.pi * D_o * 1.0           # Área do tubo exposto [m²/m]
A_1_aleta   = 2 * np.pi * (r_2**2 - r_1**2)  # Área de uma aleta [m²]
A_aletas_m  = A_1_aleta * N_aletas        # Área total de aletas por metro [m²/m]
A_total_ext = A_tubo_base + A_aletas_m    # Área externa total por metro [m²/m]

# Razão de áreas (parâmetro da correlação de Briggs & Young)
razao_areas = A_total_ext / A_tubo_base

# Coeficiente convectivo externo (Briggs & Young para feixes aletados)
Nu_o = 0.35 * Re_D**0.60 * Pr_ar**(1.0/3.0) * (razao_areas)**0.15
h_o = Nu_o * k_ar / D_o  # [W/(m²·K)]

# Parâmetro da aleta
m = np.sqrt(2 * h_o / (k_aleta * t_aleta))  # [m⁻¹]
L_c = r_2 - r_1 + t_aleta / 2               # Comprimento corrigido [m]
mL_c = m * L_c                               # Parâmetro adimensional

print("=" * 60)
print(" CRITÉRIO DE VALIDADE E CONSISTÊNCIA")
print("=" * 60)
print(f"\n📊 Escoamento do ar:")
print(f"  • Re_D = {Re_D:.0f} (faixa válida: 10³ < Re < 10⁴)")
print(f"  • {'✅ Válido' if 1e3 < Re_D < 1e4 else '⚠️ Fora da faixa'}")
print(f"\n📐 Parâmetro da aleta:")
print(f"  • m = {m:.2f} m⁻¹")
print(f"  • mL_c = {mL_c:.4f} (faixa válida: 0 < mL_c < 5)")
print(f"  • {'✅ Válido' if 0 < mL_c < 5 else '⚠️ Fora da faixa'}")
print(f"\n🌬️ Coeficiente convectivo externo:")
print(f"  • Nu_o = {Nu_o:.2f}")
print(f"  • h_o = {h_o:.2f} W/(m²·K)")
print(f"\n📏 Áreas por metro de tubo:")
print(f"  • A_tubo = {A_tubo_base:.5f} m²/m")
print(f"  • A_aletas = {A_aletas_m:.5f} m²/m")
print(f"  • A_total = {A_total_ext:.5f} m²/m")
print(f"  • Razão de ampliação = {razao_areas:.1f}×")

---

## 🌪️ Seção 5: Forçamento / Condição de Entrada

O forçamento térmico é a carga de condensação imposta pelo ciclo Rankine da usina. Para uma célula:

- **Carga térmica:** $Q = 6{,}25$ MW
- **Temperatura de saturação:** $T_{sat} = 55$ °C (constante durante a condensação)
- **Ar de resfriamento:** $T_{ar,in} = 30$ °C (projeto) ou $T_{ar,in} = 40$ °C (onda de calor)

A condição de contorno térmica é do tipo **Dirichlet** no lado do vapor (temperatura de saturação fixa) e **convectiva** no lado do ar.

In [ ]:
# ============================================================
# FORÇAMENTO TÉRMICO
# ============================================================

# Diferenças de temperatura (LMTD simplificado para condensação)
DT1 = T_sat - T_ar_in      # Diferença na entrada [K]
DT2 = T_sat - T_ar_out     # Diferença na saída [K]
LMTD = (DT1 - DT2) / np.log(DT1 / DT2)  # Média logarítmica [K]

# Capacidade térmica do ar
C_ar = mdot_ar * cp_ar  # [W/K]

print("=" * 60)
print(" FORÇAMENTO TÉRMICO")
print("=" * 60)
print(f"\n🌡️ Temperaturas:")
print(f"  • T_sat = {T_sat} °C (constante)")
print(f"  • T_ar: {T_ar_in} → {T_ar_out} °C")
print(f"\n📊 Diferenças de temperatura:")
print(f"  • ΔT₁ = {DT1:.1f} K")
print(f"  • ΔT₂ = {DT2:.1f} K")
print(f"  • LMTD = {LMTD:.2f} K")
print(f"\n⚡ Capacidade térmica do ar:")
print(f"  • C_ar = {C_ar/1e6:.3f} MW/K")

---

## 🧮 Seção 6: Implementação Numérica

### 6.1 Eficiência da Aleta (Solução de Bessel vs. Aproximação Reta)
### 6.2 Rede de Resistências Térmicas
### 6.3 Coeficiente Global e ε-NTU

In [ ]:
# ============================================================
# 6.1 EFICIÊNCIA DA ALETA — SOLUÇÃO DE BESSEL
# ============================================================

mr1 = m * r_1
mr2 = m * r_2

# Numerador da solução de Bessel
num = iv(1, mr2) * kv(1, mr1) - kv(1, mr2) * iv(1, mr1)
# Denominador da solução de Bessel
den = iv(0, mr1) * kv(1, mr2) + kv(0, mr1) * iv(1, mr2)

# Eficiência da aleta anular (solução exata)
eta_f_bessel = (2 * r_1 / (m * (r_2**2 - r_1**2))) * (num / den)

# Comparativo: aproximação de aleta reta
eta_f_reta = np.tanh(m * L_c) / (m * L_c)

print("=" * 60)
print(" 6.1 EFICIÊNCIA DA ALETA")
print("=" * 60)
print(f"\n📐 Parâmetros:")
print(f"  • mr₁ = {mr1:.4f}")
print(f"  • mr₂ = {mr2:.4f}")
print(f"\n📊 Resultados:")
print(f"  • η_f (Bessel - Exato) = {eta_f_bessel:.4f}")
print(f"  • η_f (Aleta Reta)     = {eta_f_reta:.4f}")
print(f"  • Erro relativo         = {((eta_f_reta/eta_f_bessel)-1)*100:+.2f}%")
print(f"\n💡 A aproximação de aleta reta SUPERESTIMA a eficiência.")

In [ ]:
# ============================================================
# 6.2 REDE DE RESISTÊNCIAS TÉRMICAS
# ============================================================

# Áreas totais para a célula completa
A_i_total = np.pi * D_i * L_tubo * N_t       # Área interna total [m²]
A_o_total = A_total_ext * L_tubo * N_t       # Área externa total [m²]
A_aletas_total = A_aletas_m * L_tubo * N_t   # Área de aletas total [m²]
A_tubo_exp = A_tubo_base * L_tubo * N_t      # Área de tubo exposto [m²]

# Eficiência de superfície total
eta_o = 1.0 - (A_aletas_total / A_o_total) * (1.0 - eta_f_bessel)

# Resistências térmicas [K/W]
R_int     = 1.0 / (h_i * A_i_total)                    # Convecção interna
R_fi      = Rf_i / A_i_total                           # Incrustação interna
R_parede  = np.log(D_o / D_i) / (2 * np.pi * k_tubo * L_tubo * N_t)  # Parede
R_fo      = Rf_o / A_o_total                           # Incrustação externa
R_ext     = 1.0 / (eta_o * h_o * A_o_total)            # Convecção externa

# Resistência total
R_total = R_int + R_fi + R_parede + R_fo + R_ext  # [K/W]

# Coeficiente global (referido à área externa)
U_o = 1.0 / (R_total * A_o_total)  # [W/(m²·K)]

print("=" * 60)
print(" 6.2 REDE DE RESISTÊNCIAS TÉRMICAS")
print("=" * 60)
print(f"\n📏 Áreas totais:")
print(f"  • A_i = {A_i_total:.1f} m²")
print(f"  • A_o = {A_o_total:.1f} m²")
print(f"  • A_aletas = {A_aletas_total:.1f} m²")
print(f"\n🕸️ Resistências térmicas [K/W]:")
print(f"  • R_int (condensação)  = {R_int:.6e}")
print(f"  • R_fi (incrust. int.) = {R_fi:.6e}")
print(f"  • R_parede (condução)  = {R_parede:.6e}")
print(f"  • R_fo (incrust. ext.) = {R_fo:.6e}")
print(f"  • R_ext (convecção)    = {R_ext:.6e}")
print(f"  • R_total              = {R_total:.6e}")
print(f"\n📊 Coeficiente global:")
print(f"  • U_o = {U_o:.1f} W/(m²·K)")
print(f"  • η_o = {eta_o:.4f} ({eta_o*100:.1f}%)")

# Verificação: dominância do lado do ar
pct_ext = R_ext / R_total * 100
print(f"\n✅ Dominância do lado do ar: {pct_ext:.1f}% (> 70% esperado)")

In [ ]:
# ============================================================
# 6.3 MÉTODO ε-NTU PARA CONDENSAÇÃO
# ============================================================

# NTU
NTU = U_o * A_o_total / C_ar

# Efetividade (para condensação: C_r = 0)
epsilon = 1.0 - np.exp(-NTU)

# Calor máximo transferível
Q_max = C_ar * (T_sat - T_ar_in)  # [W]

# Calor real transferido
Q_real = epsilon * Q_max  # [W]

# Temperatura de saída do ar (verificação)
T_ar_out_real = T_ar_in + Q_real / C_ar  # [°C]

print("=" * 60)
print(" 6.3 MÉTODO ε-NTU")
print("=" * 60)
print(f"\n🧮 Parâmetros:")
print(f"  • NTU = {NTU:.3f}")
print(f"  • ε = {epsilon:.4f} ({epsilon*100:.2f}%)")
print(f"\n📊 Desempenho:")
print(f"  • Q_max = {Q_max/1e6:.2f} MW")
print(f"  • Q_real = {Q_real/1e6:.2f} MW")
print(f"  • Demanda = {Q_total/1e6:.2f} MW")
print(f"  • Margem = {(Q_real - Q_total)/Q_total*100:+.1f}%")
print(f"\n🌡️ Verificação:")
print(f"  • T_ar,out (calculada) = {T_ar_out_real:.1f} °C")
print(f"  • T_ar,out (projeto)   = {T_ar_out:.1f} °C")
print(f"\n{'✅' if Q_real >= Q_total else '❌'} Carga {'atendida' if Q_real >= Q_total else 'NÃO atendida'}!")

---

## 📊 Seção 7: Visualização dos Resultados

Apresentamos a distribuição das resistências térmicas e a comparação entre modelos de aleta.

In [ ]:
# ============================================================
# VISUALIZAÇÃO: RESISTÊNCIAS E EFICIÊNCIA
# ============================================================

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# --- Gráfico 1: Distribuição de resistências (pizza) ---
resistencias = {
    'Conv. Interna': R_int,
    'Incrust. Int.': R_fi,
    'Parede': R_parede,
    'Incrust. Ext.': R_fo,
    'Conv. Ext. (Aletas)': R_ext
}
labels = list(resistencias.keys())
values = list(resistencias.values())
pct_values = [v / R_total * 100 for v in values]

cores = plt.cm.Pastel1(np.linspace(0, 1, len(labels)))
ax1.pie(pct_values, labels=labels, autopct='%1.1f%%',
        startangle=140, colors=cores)
ax1.set_title(f'Distribuição das Resistências Térmicas\n'
              f'U_o = {U_o:.1f} W/(m²·K) | η_o = {eta_o:.2%}',
              fontsize=12, fontweight='bold')

# --- Gráfico 2: Comparação de modelos de aleta ---
modelos = ['Bessel\n(Exato)', 'Aleta Reta\n(Aproximação)']
eficiencias = [eta_f_bessel, eta_f_reta]
cores_barra = ['#2196F3', '#FF9800']

barras = ax2.bar(modelos, eficiencias, color=cores_barra, width=0.5, edgecolor='black')
ax2.set_ylim(0.80, 1.0)
ax2.set_ylabel('Eficiência da Aleta (η_f)', fontsize=12)
ax2.set_title('Impacto da Modelagem Geométrica', fontsize=12, fontweight='bold')
ax2.axhline(y=eta_f_bessel, color='red', linestyle='--', alpha=0.5, label='Referência')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3, axis='y')

# Anotar valores
for barra, val in zip(barras, eficiencias):
    ax2.text(barra.get_x() + barra.get_width()/2., val + 0.002,
             f'{val:.4f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

print("\n💡 Observação:")
print("  • A resistência do lado do ar domina (> 75% do total)")
print("  • A aproximação de aleta reta superestima η_f em ~3,5%")
print("  • O uso de Bessel é mandatório para projeto preciso")

---

## 📈 Seção 8: Análise Quantitativa e Tabela Comparativa

In [ ]:
# ============================================================
# TABELA COMPARATIVA: BESSEL vs. ALETA RETA
# ============================================================

dados_comparacao = {
    "Característica": [
        "Modelo matemático",
        "Eficiência (η_f)",
        "Erro relativo",
        "η_o resultante",
        "U_o resultante [W/(m²·K)]",
        "NTU",
        "Indicação de uso"
    ],
    "Bessel (Exato)": [
        "Funções de Bessel I_n, K_n",
        f"{eta_f_bessel:.4f}",
        "Referência",
        f"{eta_o:.4f}",
        f"{U_o:.1f}",
        f"{NTU:.3f}",
        "Projeto definitivo, pós-graduação"
    ],
    "Aleta Reta (Aprox.)": [
        "tanh(mL_c) / (mL_c)",
        f"{eta_f_reta:.4f}",
        f"+{((eta_f_reta/eta_f_bessel)-1)*100:.2f}%",
        f"{1.0 - (A_aletas_total/A_o_total)*(1.0-eta_f_reta):.4f}",
        "—",
        "—",
        "Estimativa preliminar apenas"
    ]
}

df_comparacao = pd.DataFrame(dados_comparacao)

print("=" * 60)
print(" TABELA COMPARATIVA: MODELOS DE ALETA")
print("=" * 60)
display(Markdown(df_comparacao.to_markdown(index=False)))

In [ ]:
# ============================================================
# TABELA COMPARATIVA: ACC vs. CONDENSADOR A ÁGUA
# ============================================================

comparativo_ambiental = {
    'Parâmetro': [
        'Área de troca (por célula)',
        'Consumo de água',
        'Potência auxiliar',
        'Custo (CAPEX)',
        'Impacto Ambiental'
    ],
    'ACC (Ar)': [
        f'~ {A_o_total:.0f} m²',
        '0 m³/s',
        'Alta (Ventiladores)',
        'Alto (+50%)',
        'Visual e Sonoro'
    ],
    'Condensador a Água': [
        '~ 350 m²',
        '~ 3,6 m³/s',
        'Baixa (Bombas)',
        'Base',
        'Poluição Térmica no Rio'
    ]
}

df_ambiental = pd.DataFrame(comparativo_ambiental)

print("=" * 60)
print(" TRADE-OFFS: ACC vs. CONDENSADOR A ÁGUA")
print("=" * 60)
display(Markdown(df_ambiental.to_markdown(index=False)))

---

## 🔍 Seção 9: Análise de Sensibilidade — Onda de Calor

Como o ACC reage a um verão extremo ($T_{ar} = 40$ °C)? A usina consegue manter os 100 MW$_e$?

> 💡 **Hipótese de modelagem:** Mantemos a vazão de ar e o $h_o$ constantes para isolar o efeito puramente termodinâmico da redução da LMTD.

In [ ]:
# ============================================================
# ANÁLISE DE SENSIBILIDADE: TEMPERATURA AMBIENTE
# ============================================================

T_ar_range = np.linspace(20, 45, 50)
W_range = []       # Potência elétrica máxima [MW]
LMTD_range = []    # LMTD [K]
eta_ciclo = 0.40   # Eficiência do ciclo Rankine

for T in T_ar_range:
    DT1_t = T_sat - T
    if DT1_t <= 0:
        W_range.append(0)
        LMTD_range.append(0)
        continue

    # Iterar para encontrar T_out que fecha Q = U*A*LMTD = mdot*cp*(T_out-T_in)
    T_out_guess = T + 10.0
    for _ in range(10):
        DT2_t = T_sat - T_out_guess
        if DT2_t <= 0.1:
            T_out_guess = T_sat - 0.1
            DT2_t = 0.1
            break
        LMTD_t = (DT1_t - DT2_t) / np.log(DT1_t / DT2_t)
        Q_t = U_o * A_o_total * LMTD_t
        T_out_guess = T + Q_t / C_ar

    DT2_t = T_sat - T_out_guess
    if DT2_t > 0:
        LMTD_range.append(LMTD_t)
        W_range.append(eta_ciclo * Q_t / 1e6)  # Potência elétrica [MW]
    else:
        LMTD_range.append(0)
        W_range.append(0)

# --- Gráfico de sensibilidade ---
fig, ax1 = plt.subplots(figsize=(12, 6))
color1 = 'tab:red'
ax1.set_xlabel('Temperatura do Ar [°C]', fontsize=12)
ax1.set_ylabel('Potência Elétrica Máxima [MW]', fontsize=12, color=color1)
ax1.plot(T_ar_range, W_range, color=color1, linewidth=2.5, marker='o', markersize=3)
ax1.tick_params(axis='y', labelcolor=color1)
ax1.axhline(y=100, color='g', linestyle='--', linewidth=2, label='Meta: 100 MW')
ax1.axvline(x=30, color='gray', linestyle=':', linewidth=1.5, label='Projeto (30°C)')
ax1.axvline(x=40, color='orange', linestyle=':', linewidth=1.5, label='Onda de Calor (40°C)')
ax1.legend(loc='upper right', fontsize=11)
ax1.grid(True, alpha=0.3)

ax2 = ax1.twinx()
color2 = 'tab:blue'
ax2.set_ylabel('LMTD [K]', fontsize=12, color=color2)
ax2.plot(T_ar_range, LMTD_range, color=color2, linewidth=2.5, linestyle='--')
ax2.tick_params(axis='y', labelcolor=color2)

plt.title('Impacto da Temperatura Ambiente no Desempenho do ACC',
          fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Resultados nos pontos de interesse
idx_30 = np.argmin(np.abs(T_ar_range - 30))
idx_40 = np.argmin(np.abs(T_ar_range - 40))
print(f"\n📊 Resultados da sensibilidade:")
print(f"  • T_ar = 30°C → W = {W_range[idx_30]:.1f} MW")
print(f"  • T_ar = 40°C → W = {W_range[idx_40]:.1f} MW")
print(f"  • Perda por onda de calor: {(1 - W_range[idx_40]/W_range[idx_30])*100:.1f}%")

---

## 💡 Seção 10: Discussão, Limitações e Aplicações Práticas

### Resultados Principais

1. **Dominância térmica:** A resistência do lado do ar (mesmo aletado) representa > 75% da resistência total. O projeto de ACC é, essencialmente, um desafio de otimização do lado externo.

2. **Rigor matemático:** O uso de Funções de Bessel para aletas anulares é mandatório em nível de pós-graduação. A aproximação unidimensional reta superestima a transferência de calor em ~3,5%.

3. **Vulnerabilidade climática:** Ondas de calor reduzem a LMTD e, consequentemente, a capacidade de rejeição de calor, forçando o *derating* da usina.

4. **Método ε-NTU:** Para condensação ($C_r = 0$), a efetividade tem forma exponencial simples, facilitando a verificação rápida de desempenho.

### Limitações do Modelo

- **h_i constante:** O coeficiente de condensação foi assumido como 8000 W/(m²·K). Na prática, ele depende da diferença $T_{sat} - T_{parede}$ e exige iteração.
- **Correlação de Briggs & Young:** Válida para faixas específicas de $Re_D$ e geometria. Fora dessas faixas, o erro pode ser significativo.
- **Propriedades constantes:** As propriedades do ar foram avaliadas na temperatura média. Para grandes $\Delta T$, propriedades variáveis seriam mais precisas.
- **Geometria idealizada:** Não considera efeitos de borda, escoamento não-uniforme entre tubos, ou bloqueio parcial das aletas.

### Aplicações Práticas

- **Usinas em regiões áridas:** O ACC elimina o consumo de água, sendo a única opção viável em regiões de escassez hídrica.
- **Sistemas híbridos:** Em ondas de calor, a aspersão de água sobre as aletas pode reduzir $T_{ar,in}$ em 10–12°C, recuperando a potência.
- **Gestão ambiental:** Ao eliminar a descarga térmica em rios, o ACC preserva ecossistemas aquáticos.

> 📖 **Referência:** Incropera et al. (2017), Cap. 3 e 11; Gobbi (s.d.), Cap. 3 — Balanço de energia.

---

## 🔄 Navegação

- [📚 Voltar ao Capítulo 10](../notebooks/10_Aletas_Superficies_Estendidas.ipynb)
- [📂 Outros Estudos de Caso](README.md)
- [🏠 Repositório Principal](../README.md)

---

<div align="center">

Estudo de Caso 10.5
Parte da coleção *"Fenômenos de Transporte: Fundamentos e Modelagem Computacional"*

</div>